# Pipeline de Agentes: qualificação de leads e e-mail de vendas com CrewAI

Neste notebook vamos montar um **pipeline comercial feito só com agentes de IA**. Dado um lead (uma pessoa que demonstrou interesse no nosso produto), o sistema:

1. **Pesquisa o lead na internet**: quem é a pessoa, o que a empresa faz, porte, setor, presença de mercado.
2. **Avalia se a empresa combina com a nossa** (afinidade cultural e estratégica).
3. **Dá uma nota de 0 a 100** para o lead e valida essa nota.
4. **Filtra** só os leads bons (nota acima de 70).
5. **Escreve um e-mail de vendas personalizado** para cada lead aprovado e depois **otimiza** esse e-mail para gerar resposta.

Ao todo são **5 agentes**, organizados em **2 equipes (crews)** e coordenados por um **Flow**:

```
                 ┌──────────────── Crew 1: Qualificação de Leads ────────────────┐
buscar_leads ──► │ Especialista em Dados ─► Analista de Afinidade ─► Avaliador   │ ──► filtrar_leads ──┐
                 └─────────── (usam busca no Google + leitura de sites) ─────────┘                     │
                                                                                                        ▼
                                  ┌────────── Crew 2: Engajamento por E-mail ──────────┐
                  enviar_email ◄─ │ Redator de E-mails ─► Especialista em Engajamento │ ◄── escrever_email
                                  └────────────────────────────────────────────────────┘
```

### Conceitos do CrewAI que aparecem aqui

| Conceito | O que é |
|---|---|
| **Agent (Agente)** | Um "funcionário" de IA com papel (`role`), objetivo (`goal`) e história (`backstory`). Usa um LLM para pensar. |
| **Task (Tarefa)** | O trabalho que um agente precisa fazer, com descrição e resultado esperado. |
| **Tool (Ferramenta)** | Algo que o agente pode usar além do LLM, como buscar no Google ou ler um site. |
| **Crew (Equipe)** | Um grupo de agentes executando uma sequência de tarefas. |
| **Flow (Fluxo)** | Orquestra várias etapas (e várias crews) com código Python, decidindo o que roda depois de quê. |

> Este material é uma adaptação, traduzida e comentada, de uma aula do curso *Practical Multi AI Agents and Advanced Use Cases with crewAI* (DeepLearning.AI + CrewAI).

## 0. Como rodar este notebook

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pasteurjr/notebookvideoaulas/blob/main/05_agentes_crewai/notebook/pipeline_de_agentes.ipynb)

O curso tem duas partes, com a **mesma estrutura de pastas** (uma pasta por aula):

| O quê | Onde |
|---|---|
| **Código**: os notebooks e os arquivos que eles usam | GitHub: **https://github.com/pasteurjr/notebookvideoaulas** |
| **Dados, vídeos e roteiros** de cada aula | Google Drive: **[pasta notebooksvideos](https://drive.google.com/drive/folders/1mMRjinjohj_jyMsJzO7rxgYKm6unVKze?usp=sharing)** |

**Antes de começar, leia o [README.md da pasta do Google Drive](https://drive.google.com/drive/folders/1mMRjinjohj_jyMsJzO7rxgYKm6unVKze?usp=sharing)**: ele explica como baixar e organizar os dados de todas as aulas.

**Esta aula (`05_agentes_crewai`) usa:** nenhum arquivo de dados (as configurações dos agentes já vêm no GitHub); não precisa de GPU; chaves de API: `DEEPSEEK_API_KEY`, `SERPER_API_KEY`.

### Opção A: Google Colab (recomendado)

1. **Abra este notebook no Colab** pelo botão **Abrir no Colab** acima (ele abre direto do GitHub).
2. **Guarde as chaves de API nos *Secrets* do Colab:** clique no ícone de **chave (🔑)** na barra lateral, crie `DEEPSEEK_API_KEY` e `SERPER_API_KEY` com as **suas** chaves (com crédito) e ative **Acesso ao notebook**. As chaves ficam na sua conta Google, nunca no notebook.
3. **Rode a célula de preparação logo abaixo.** Ela clona o repositório do GitHub para o Colab, carrega as chaves dos *Secrets* e instala só as bibliotecas que faltarem.
4. Depois: **Ambiente de execução → Executar tudo**.

**Se aparecer erro de biblioteca** (`ModuleNotFoundError`, `ImportError` ou erro de versão): crie uma célula nova, rode o comando abaixo, depois vá em **Ambiente de execução → Reiniciar sessão** e rode a célula de preparação de novo.

```
!pip install crewai==1.15.22 crewai-tools==1.15.22 python-dotenv pyyaml
```

### Opção B: no seu computador

Basta baixar. Pré-requisito: [Miniconda](https://docs.conda.io/en/latest/miniconda.html) (ou Anaconda) e Git. No terminal:

```bash
git clone https://github.com/pasteurjr/notebookvideoaulas.git
cd notebookvideoaulas
conda create -n notecursos python=3.12 -y
conda activate notecursos
pip install -r requirements.txt
cd 05_agentes_crewai/notebook
jupyter notebook pipeline_de_agentes.ipynb
```


**Chaves:** copie `.env.exemplo` para `.env` e coloque nele as **suas** chaves (`DEEPSEEK_API_KEY`, `SERPER_API_KEY`). O `.env` nunca vai para o GitHub.

No computador local, a célula de preparação detecta que não está no Colab e não faz nada.

### Dica: peça ajuda a um assistente de IA

Um assistente de programação, como o **Claude Code** ou o **Codex**, faz toda essa preparação sozinho. Abra o assistente numa pasta vazia e peça, por exemplo:

> Clone https://github.com/pasteurjr/notebookvideoaulas. Crie o ambiente conda `notecursos` com Python 3.12 e instale: `pip install -r requirements.txt`. Copie `.env.exemplo` para `.env` e me peça as chaves de API. Por fim, abra `05_agentes_crewai/notebook/pipeline_de_agentes.ipynb` no Jupyter.


In [ ]:
# PREPARAÇÃO DO AMBIENTE NO GOOGLE COLAB (igual em todos os notebooks do curso; no computador local não faz nada)
AULA = "05_agentes_crewai/notebook"                           # pasta desta aula no repositório
USA_DADOS_DO_DRIVE = False                                     # copia os dados da pasta "notebooksvideos" do Drive
PACOTES = {"crewai": "crewai==1.15.22", "crewai_tools": "crewai-tools==1.15.22", "dotenv": "python-dotenv", "yaml": "pyyaml"}   # módulo: pacote pip
SECRETS = ["DEEPSEEK_API_KEY", "SERPER_API_KEY"]                                   # chaves lidas dos Secrets do Colab (ícone 🔑)

import os, sys, shutil, subprocess, importlib.util

if "google.colab" in sys.modules:
    # 1) Código: clona o repositório do GitHub para o Colab
    if not os.path.exists("/content/notebookvideoaulas"):
        subprocess.run(["git", "clone", "-q", "https://github.com/pasteurjr/notebookvideoaulas.git",
                        "/content/notebookvideoaulas"], check=True)
    os.chdir(f"/content/notebookvideoaulas/{AULA}")

    # 2) Dados: copia da pasta "notebooksvideos" (atalho no Meu Drive) para a pasta da aula
    if USA_DADOS_DO_DRIVE and not os.path.exists("dados"):
        from google.colab import drive
        drive.mount("/content/drive")
        origem = f"/content/drive/MyDrive/notebooksvideos/{AULA}/dados"
        if not os.path.exists(origem):
            raise FileNotFoundError(f"Não encontrei {origem}. Coloque a pasta notebooksvideos do curso no seu Meu Drive "
                                    "(veja o README.md da pasta do Google Drive) e rode esta célula de novo.")
        shutil.copytree(origem, "dados")

    # 3) Bibliotecas: instala só as que a sessão do Colab ainda não tem
    faltando = [pip for modulo, pip in PACOTES.items() if importlib.util.find_spec(modulo) is None]
    if faltando:
        print("Instalando:", ", ".join(faltando))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltando], check=True)
        print("Pronto. Se o Colab pedir 'Reiniciar sessão', reinicie e rode esta célula de novo.")
    else:
        print("Todas as bibliotecas já estão instaladas no Colab.")

    # 4) Chaves de API: vêm dos Secrets do Colab (nunca escreva a chave no notebook)
    if SECRETS:
        from google.colab import userdata
        for chave in SECRETS:
            os.environ[chave] = userdata.get(chave)
    os.environ.setdefault("MODELO_LLM", "deepseek/deepseek-v4-flash")

    print("Pasta da aula:", os.getcwd())
    print("Conteúdo:", sorted(os.listdir(".")))
else:
    print("Ambiente local detectado: usando a pasta atual.")

## 1. Pré-requisitos: ambiente e chaves de API

### Ambiente Python
O notebook foi testado com Python 3.12 e CrewAI 1.15, no ambiente Conda `notecursos` (veja a seção 0 para criar o ambiente ou para rodar no Google Colab).

### Chaves de API: **cada aluno precisa criar as suas, com crédito**
Os agentes usam dois serviços pagos. **Sem uma chave válida e com saldo, o notebook não funciona.**

| Serviço | Para que serve | Onde criar a chave |
|---|---|---|
| **DeepSeek** | É o LLM, o "cérebro" dos agentes | https://platform.deepseek.com/api_keys (é preciso colocar crédito na conta; alguns dólares bastam para muitas execuções) |
| **Serper** | Faz buscas no Google para os agentes | https://serper.dev/api-key (a conta nova vem com créditos gratuitos de busca) |

As chaves **nunca** devem ficar escritas no notebook: se o notebook for compartilhado, a chave vaza junto. Por isso elas ficam num arquivo separado chamado **`.env`**, na mesma pasta deste notebook:

1. Copie o arquivo `.env.exemplo` para `.env`.
2. Abra o `.env` e troque `coloque_sua_chave_aqui` pelas suas chaves.
3. Não envie o `.env` para ninguém nem para o GitHub (ele já está no `.gitignore`).

> **Quer usar outro LLM?** O CrewAI aceita vários provedores. Basta mudar a variável `MODELO_LLM` no `.env`, por exemplo `openai/gpt-4o-mini` (com `OPENAI_API_KEY`) ou `anthropic/claude-sonnet-4-5` (com `ANTHROPIC_API_KEY`).

## 2. Conferindo as versões instaladas

Primeiro verificamos se as bibliotecas principais estão instaladas e em que versão. Isso ajuda a diagnosticar problemas: o CrewAI muda bastante entre versões.

In [1]:
import crewai
import pydantic

print(f"Versão do CrewAI:   {crewai.__version__}")
print(f"Versão do Pydantic: {pydantic.__version__}")

Versão do CrewAI:   1.15.22
Versão do Pydantic: 2.12.5


## 3. Importações e carregamento do `.env`

- `load_dotenv(".env")` lê o arquivo `.env` e coloca cada chave nas variáveis de ambiente do Python (`os.environ`). As ferramentas do CrewAI procuram as chaves ali.
- `Agent`, `Task`, `Crew` e `LLM` são as peças principais do CrewAI.
- `yaml` serve para ler os arquivos de configuração dos agentes e tarefas.

In [2]:
# Esconde avisos que não interessam para a aula
import warnings
warnings.filterwarnings('ignore')

import os
import yaml
from dotenv import load_dotenv

# Lê as chaves do arquivo .env (que fica na mesma pasta do notebook)
load_dotenv(".env")

from crewai import Agent, Task, Crew, LLM

## 4. Verificando as chaves e configurando o LLM

Esta célula **não mostra** as chaves. Ela só confere se elas foram carregadas do `.env`; se faltar alguma, avisa o que fazer.

Depois criamos o objeto `llm`, que é o modelo de linguagem usado por todos os agentes. O nome do modelo vem da variável `MODELO_LLM` do `.env` (por padrão, `deepseek/deepseek-v4-flash`: o prefixo antes da barra diz ao CrewAI qual é o provedor).

In [3]:
chaves_necessarias = ["DEEPSEEK_API_KEY", "SERPER_API_KEY"]
faltando = [chave for chave in chaves_necessarias
            if not os.getenv(chave) or "coloque_sua_chave" in os.getenv(chave)]

if faltando:
    raise RuntimeError(
        f"Chaves não configuradas: {faltando}. "
        "Copie .env.exemplo para .env e preencha com as SUAS chaves (com crédito)."
    )
print("Chaves carregadas do .env:", ", ".join(chaves_necessarias))

# O "cérebro" dos agentes
llm = LLM(
    model=os.getenv("MODELO_LLM", "deepseek/deepseek-v4-flash"),
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    temperature=0.3,
)
print("Modelo em uso:", llm.model)

Chaves carregadas do .env: DEEPSEEK_API_KEY, SERPER_API_KEY
Modelo em uso: deepseek-v4-flash


## 5. Carregando agentes e tarefas dos arquivos YAML

Em vez de escrever os textos dos agentes e tarefas no meio do código, o CrewAI permite guardá-los em arquivos **YAML** na pasta `config/`. Isso deixa o código limpo e facilita ajustar os textos (*prompts*) sem mexer no Python.

| Arquivo | Conteúdo |
|---|---|
| `agentes_qualificacao_leads.yaml` | Os 3 agentes que pesquisam e pontuam o lead |
| `tarefas_qualificacao_leads.yaml` | As 3 tarefas desses agentes |
| `agentes_engajamento_email.yaml` | Os 2 agentes que escrevem o e-mail |
| `tarefas_engajamento_email.yaml` | As 2 tarefas desses agentes |

**YAML** é um formato de texto simples para configurações, organizado em pares `nome: valor`. Cada arquivo traz estes campos:

| Campo | Arquivo | Para que serve |
|---|---|---|
| `role` | agentes | O **papel** do agente (o "cargo" dele na equipe) |
| `goal` | agentes | O **objetivo** que ele persegue |
| `backstory` | agentes | A **história**: experiência e jeito de trabalhar, que dão contexto e personalidade ao agente |
| `verbose` / `allow_delegation` | agentes | Mostrar o raciocínio na tela / permitir repassar trabalho a outro agente (aqui, não) |
| `description` | tarefas | **O que** deve ser feito, passo a passo |
| `expected_output` | tarefas | **Como** deve ser a entrega |

Tudo isso vira instrução (*prompt*) para o LLM. Os trechos entre chaves, como `{dados_lead}`, são **variáveis** preenchidas na hora da execução.

In [4]:
# Caminhos dos arquivos de configuração
arquivos = {
    'agentes_leads':  'config/agentes_qualificacao_leads.yaml',
    'tarefas_leads':  'config/tarefas_qualificacao_leads.yaml',
    'agentes_email':  'config/agentes_engajamento_email.yaml',
    'tarefas_email':  'config/tarefas_engajamento_email.yaml',
}

# Lê cada YAML e guarda num dicionário
configs = {}
for tipo, caminho in arquivos.items():
    with open(caminho, 'r', encoding='utf-8') as arquivo:
        configs[tipo] = yaml.safe_load(arquivo)

config_agentes_leads = configs['agentes_leads']
config_tarefas_leads = configs['tarefas_leads']
config_agentes_email = configs['agentes_email']
config_tarefas_email = configs['tarefas_email']

print("Agentes de qualificação:", list(config_agentes_leads))
print("Agentes de e-mail:      ", list(config_agentes_email))

Agentes de qualificação: ['agente_dados_lead', 'agente_fit_cultural', 'agente_pontuacao_validacao']
Agentes de e-mail:       ['especialista_conteudo_email', 'estrategista_engajamento']


## 6. Modelos Pydantic: forçando uma saída estruturada

Normalmente um LLM devolve texto livre, e texto livre é difícil de usar num programa. O **Pydantic** é uma biblioteca Python para **descrever e validar estruturas de dados**: com ele definimos o **formato exato** que queremos na resposta, ou seja, quais campos, de que tipo e com que limites (por exemplo, nota de 0 a 10).

Depois, ao criar a tarefa final da crew de qualificação, usamos `output_pydantic=ResultadoQualificacaoLead`. O CrewAI então obriga o agente a responder nesse formato, e conseguimos acessar, por exemplo, `resultado.pontuacao_lead.pontuacao` como um número de verdade.

A estrutura tem três blocos:
- `DadosPessoaisLead`: quem é a pessoa;
- `DadosEmpresa`: a empresa onde ela trabalha;
- `PontuacaoLead`: a nota final, os critérios usados e as observações da validação.

In [5]:
from pydantic import BaseModel, Field
from typing import List, Optional

class DadosPessoaisLead(BaseModel):
    nome: str = Field(..., description="Nome completo do lead.")
    cargo: str = Field(..., description="Cargo atual do lead.")
    relevancia_cargo: int = Field(..., ge=0, le=10, description="Nota de 0 a 10 para o peso do cargo na decisão de compra.")
    historico_profissional: Optional[str] = Field(..., description="Breve resumo da trajetória profissional do lead.")

class DadosEmpresa(BaseModel):
    nome_empresa: str = Field(..., description="Nome da empresa onde o lead trabalha.")
    setor: str = Field(..., description="Setor de atuação da empresa.")
    porte: int = Field(..., description="Porte da empresa em número de funcionários.")
    faturamento: Optional[float] = Field(None, description="Faturamento anual da empresa, se disponível.")
    presenca_mercado: int = Field(..., ge=0, le=10, description="Nota de 0 a 10 para a presença de mercado da empresa.")

class PontuacaoLead(BaseModel):
    pontuacao: int = Field(..., ge=0, le=100, description="Nota final do lead (0 a 100).")
    criterios: List[str] = Field(..., description="Critérios usados para chegar à nota.")
    observacoes_validacao: Optional[str] = Field(None, description="Observações sobre a validação da nota.")

class ResultadoQualificacaoLead(BaseModel):
    dados_pessoais: DadosPessoaisLead = Field(..., description="Informações pessoais do lead.")
    dados_empresa: DadosEmpresa = Field(..., description="Informações da empresa do lead.")
    pontuacao_lead: PontuacaoLead = Field(..., description="Nota calculada e informações relacionadas.")

## 7. Ferramentas: dando "mãos" aos agentes

Sem ferramentas, o agente só sabe o que o LLM aprendeu no treinamento. Com elas, ele **busca informação atualizada na internet**:

- **`SerperDevTool`**: faz buscas no Google (via Serper) e devolve os resultados. Usa a `SERPER_API_KEY` do `.env`.
- **`ScrapeWebsiteTool`**: abre uma página da web e extrai o texto, por exemplo o site da empresa do lead.

O agente decide **sozinho** quando e como usar cada ferramenta, de acordo com a tarefa.

In [6]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

ferramenta_busca = SerperDevTool()         # busca no Google
ferramenta_leitura = ScrapeWebsiteTool()   # lê o conteúdo de sites

## 8. Crew 1: qualificação de leads (3 agentes, 3 tarefas)

Aqui montamos a primeira equipe:

| Agente | Tarefa | Ferramentas |
|---|---|---|
| Especialista em Dados de Leads | Levantar dados da pessoa e da empresa | busca + leitura de sites |
| Analista de Afinidade Cultural | Avaliar se a empresa combina com a nossa | busca + leitura de sites |
| Avaliador e Validador de Leads | Dar a nota final e validar | busca + leitura de sites |

Pontos importantes:
- `config=...` pega o papel, objetivo e história do YAML.
- `llm=llm` define o modelo que o agente usa.
- Na terceira tarefa, `context=[tarefa_dados_lead, tarefa_fit_cultural]` faz o avaliador **receber o resultado das duas tarefas anteriores**. É assim que os agentes "conversam".
- `output_pydantic=ResultadoQualificacaoLead` obriga a saída final a seguir o modelo Pydantic da seção 6.
- `verbose=True` mostra o "raciocínio" dos agentes durante a execução, o que é ótimo para a aula.

In [7]:
# --- Agentes ---
agente_dados_lead = Agent(
    config=config_agentes_leads['agente_dados_lead'],
    tools=[ferramenta_busca, ferramenta_leitura],
    llm=llm,
)

agente_fit_cultural = Agent(
    config=config_agentes_leads['agente_fit_cultural'],
    tools=[ferramenta_busca, ferramenta_leitura],
    llm=llm,
)

agente_pontuacao_validacao = Agent(
    config=config_agentes_leads['agente_pontuacao_validacao'],
    tools=[ferramenta_busca, ferramenta_leitura],
    llm=llm,
)

# --- Tarefas ---
tarefa_dados_lead = Task(
    config=config_tarefas_leads['coleta_dados_lead'],
    agent=agente_dados_lead,
)

tarefa_fit_cultural = Task(
    config=config_tarefas_leads['analise_fit_cultural'],
    agent=agente_fit_cultural,
)

tarefa_pontuacao_validacao = Task(
    config=config_tarefas_leads['pontuacao_e_validacao_lead'],
    agent=agente_pontuacao_validacao,
    context=[tarefa_dados_lead, tarefa_fit_cultural],   # recebe o resultado das duas anteriores
    output_pydantic=ResultadoQualificacaoLead,          # saída estruturada
)

# --- Equipe ---
crew_qualificacao_leads = Crew(
    agents=[agente_dados_lead, agente_fit_cultural, agente_pontuacao_validacao],
    tasks=[tarefa_dados_lead, tarefa_fit_cultural, tarefa_pontuacao_validacao],
    verbose=True,
)

## 9. Crew 2: engajamento por e-mail (2 agentes, 2 tarefas)

A segunda equipe recebe o lead já qualificado e escreve o e-mail:

1. O **Redator de E-mails** escreve um rascunho personalizado com os dados pesquisados.
2. O **Especialista em Engajamento** revisa o rascunho e acrescenta chamadas para ação (CTAs), como "agende uma conversa de 20 minutos".

Esses agentes **não precisam de ferramentas**: todas as informações já vêm prontas da crew anterior. As tarefas rodam em sequência, e a segunda recebe automaticamente o resultado da primeira.

In [8]:
# --- Agentes ---
especialista_conteudo_email = Agent(
    config=config_agentes_email['especialista_conteudo_email'],
    llm=llm,
)

estrategista_engajamento = Agent(
    config=config_agentes_email['estrategista_engajamento'],
    llm=llm,
)

# --- Tarefas ---
tarefa_redacao_email = Task(
    config=config_tarefas_email['redacao_email'],
    agent=especialista_conteudo_email,
)

tarefa_otimizacao_engajamento = Task(
    config=config_tarefas_email['otimizacao_engajamento'],
    agent=estrategista_engajamento,
)

# --- Equipe ---
crew_redacao_email = Crew(
    agents=[especialista_conteudo_email, estrategista_engajamento],
    tasks=[tarefa_redacao_email, tarefa_otimizacao_engajamento],
    verbose=True,
)

## 10. O Flow: juntando tudo num pipeline de vendas

Um **Flow** é uma classe Python em que cada método é uma etapa do processo. Os **decoradores** dizem a ordem:

- `@start()`: a etapa que começa o fluxo.
- `@listen(etapa)`: roda quando a `etapa` indicada termina, recebendo o resultado dela.

Nosso fluxo:

| Etapa | O que faz |
|---|---|
| `buscar_leads` | Devolve a lista de leads. Aqui é uma lista fixa de exemplo; num sistema real viria do banco de dados ou do CRM. |
| `pontuar_leads` | Roda a crew de qualificação **para cada lead** (`kickoff_for_each`) e guarda os resultados no **estado** do fluxo (`self.state`). |
| `armazenar_pontuacoes` | Lugar para salvar as notas no banco de dados (aqui só devolve). |
| `filtrar_leads` | Mantém só os leads com nota **maior que 70**. |
| `escrever_email` | Roda a crew de e-mail para cada lead aprovado. |
| `enviar_email` | Lugar para enviar os e-mails de verdade (aqui só devolve). |

> **Sobre o lead de exemplo:** a pessoa (**Carla Mendes**) e o e-mail são **fictícios**. A empresa é real, para que os agentes tenham o que pesquisar na internet. Troque pelos dados que quiser.

In [9]:
from crewai import Flow
from crewai.flow.flow import listen, start

class PipelineVendas(Flow):

    @start()
    def buscar_leads(self):
        # Num sistema real, os leads viriam do banco de dados / CRM
        leads = [
            {
                "dados_lead": {
                    "nome": "Carla Mendes (pessoa fictícia)",
                    "cargo": "Diretora de Engenharia",
                    "empresa": "Nubank",
                    "email": "carla.mendes@exemplo.com.br",
                    "caso_de_uso": "Usar agentes de IA para automatizar o atendimento ao cliente e a análise de dados.",
                },
            },
        ]
        return leads

    @listen(buscar_leads)
    def pontuar_leads(self, leads):
        resultados = crew_qualificacao_leads.kickoff_for_each(leads)
        self.state["resultados_qualificacao"] = resultados
        return resultados

    @listen(pontuar_leads)
    def armazenar_pontuacoes(self, resultados):
        # Aqui salvaríamos as notas no banco de dados
        return resultados

    @listen(pontuar_leads)
    def filtrar_leads(self, resultados):
        return [r for r in resultados if r['pontuacao_lead'].pontuacao > 70]

    @listen(filtrar_leads)
    def escrever_email(self, leads):
        leads_pontuados = [lead.to_dict() for lead in leads]
        emails = crew_redacao_email.kickoff_for_each(leads_pontuados)
        return emails

    @listen(escrever_email)
    def enviar_email(self, emails):
        # Aqui enviaríamos os e-mails para os leads
        return emails

fluxo = PipelineVendas()

## 11. Visualizando o fluxo

O CrewAI desenha o fluxo num arquivo HTML interativo: cada caixa é uma etapa e as setas mostram quem escuta quem.

O método `fluxo.plot()` gera três arquivos (HTML, CSS e JavaScript) numa pasta temporária do sistema. A função `mostrar_fluxo` abaixo copia esses arquivos para a pasta `diagramas/` do projeto, junta os três num documento só e exibe o desenho dentro do notebook. Você também pode abrir `diagramas/fluxo_pipeline_vendas.html` direto no navegador.

In [10]:
import html
import shutil
from pathlib import Path
from IPython.display import HTML

def mostrar_fluxo(fluxo, nome):
    # Gera o diagrama do fluxo e copia os arquivos (.html, _style.css, _script.js) para ./diagramas
    html_temporario = Path(fluxo.plot(f"{nome}.html", show=False))
    pasta = Path("diagramas")
    pasta.mkdir(exist_ok=True)
    for arquivo in html_temporario.parent.glob(f"{nome}*"):
        shutil.copy(arquivo, pasta / arquivo.name)

    # Junta HTML + CSS + JavaScript num documento único para exibir dentro do notebook
    pagina = (pasta / f"{nome}.html").read_text(encoding="utf-8")
    css = (pasta / f"{nome}_style.css").read_text(encoding="utf-8")
    js = (pasta / f"{nome}_script.js").read_text(encoding="utf-8")
    pagina = pagina.replace(f'<link rel="stylesheet" href="{nome}_style.css" />', f"<style>{css}</style>")
    pagina = pagina.replace(f'<script src="{nome}_script.js"></script>', f"<script>{js}</script>")
    return HTML(f'<iframe srcdoc="{html.escape(pagina)}" width="100%" height="750" style="border:none"></iframe>')

mostrar_fluxo(fluxo, "fluxo_pipeline_vendas")

## 12. Executando o pipeline

Agora rodamos o fluxo inteiro. Como o Jupyter já tem um "laço assíncrono" em execução, usamos `await fluxo.kickoff_async()`.

**Isso leva alguns minutos.** Com `verbose=True` você vai ver cada agente:
- recebendo a tarefa,
- decidindo buscar no Google ou ler um site,
- analisando os resultados,
- entregando a resposta final.

Cada execução gasta créditos do LLM e das buscas; o custo é pequeno, mas não é zero.

In [11]:
emails = await fluxo.kickoff_async()

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: PipelineVendas                                                                                           │
│  ID: 8016527d-2e6c-4851-9e2b-ffaeeb006985                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: PipelineVendas                                                                                           │
│  ID: 8016527d-2e6c-4851-9e2b-ffaeeb006985                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: buscar_leads                                                                                           │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: buscar_leads                                                                                           │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: pontuar_leads                                                                                          │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: escrever_email                                                                                         │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: enviar_email                                                                                           │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: enviar_email                                                                                           │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: PipelineVendas                                                                                           │
│  ID: 8016527d-2e6c-4851-9e2b-ffaeeb006985                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 13. Uso de tokens e custo

Todo LLM cobra por **token** (pedaços de palavras), tanto do texto enviado (*prompt*) quanto do texto gerado (*completion*). O CrewAI soma o consumo de cada crew em `token_usage`.

Abaixo transformamos esses números numa tabela e estimamos o custo. O **preço por milhão de tokens é um valor de exemplo**: confira a tabela atual do provedor que você estiver usando e ajuste a variável.

In [12]:
import pandas as pd

# Preço de EXEMPLO em dólares por 1 milhão de tokens: ajuste conforme o seu provedor
PRECO_POR_MILHAO_TOKENS_USD = 0.15

# Consumo da crew de qualificação (primeiro lead)
uso_qualificacao = pd.DataFrame([fluxo.state["resultados_qualificacao"][0].token_usage.model_dump()])

custo = PRECO_POR_MILHAO_TOKENS_USD * uso_qualificacao['total_tokens'].sum() / 1_000_000
print(f"Crew de qualificação: custo estimado de US$ {custo:.4f}")
uso_qualificacao

Crew de qualificação: custo estimado de US$ 0.0247


,total_tokens,prompt_tokens,cached_prompt_tokens,completion_tokens,reasoning_tokens,cache_creation_tokens,successful_requests
0,164667,135882,54528,28785,6708,0,21


In [13]:
# Consumo da crew de e-mail (primeiro lead aprovado)
if emails:
    uso_email = pd.DataFrame([emails[0].token_usage.model_dump()])
    custo = PRECO_POR_MILHAO_TOKENS_USD * uso_email['total_tokens'].sum() / 1_000_000
    print(f"Crew de e-mail: custo estimado de US$ {custo:.4f}")
    display(uso_email)
else:
    print("Nenhum lead passou do filtro (nota > 70), então a crew de e-mail não rodou.")

Crew de e-mail: custo estimado de US$ 0.0179


,total_tokens,prompt_tokens,cached_prompt_tokens,completion_tokens,reasoning_tokens,cache_creation_tokens,successful_requests
0,119638,97110,37120,22528,6184,0,18


## 14. Inspecionando a qualificação do lead

Como a tarefa final usou `output_pydantic`, o resultado está em `.pydantic` como um objeto Python com campos. Montamos uma tabela com tudo o que os agentes descobriram e com a nota final.

In [14]:
from IPython.display import display, HTML

resultados_qualificacao = fluxo.state["resultados_qualificacao"]
qualificacao = resultados_qualificacao[0].pydantic

dados = {
    'Nome':                   qualificacao.dados_pessoais.nome,
    'Cargo':                  qualificacao.dados_pessoais.cargo,
    'Relevância do cargo':    qualificacao.dados_pessoais.relevancia_cargo,
    'Histórico profissional': qualificacao.dados_pessoais.historico_profissional,
    'Empresa':                qualificacao.dados_empresa.nome_empresa,
    'Setor':                  qualificacao.dados_empresa.setor,
    'Porte (funcionários)':   qualificacao.dados_empresa.porte,
    'Faturamento':            qualificacao.dados_empresa.faturamento,
    'Presença de mercado':    qualificacao.dados_empresa.presenca_mercado,
    'Nota do lead':           qualificacao.pontuacao_lead.pontuacao,
    'Critérios':              ', '.join(qualificacao.pontuacao_lead.criterios),
    'Observações':            qualificacao.pontuacao_lead.observacoes_validacao,
}

tabela = pd.DataFrame.from_dict(dados, orient='index', columns=['Valor']).reset_index()
tabela = tabela.rename(columns={'index': 'Atributo'})

tabela_html = (tabela.style
        .set_properties(**{'text-align': 'left'})
        .format({'Atributo': lambda x: f'<b>{x}</b>'})
        .hide(axis='index')
        .to_html())
# As classes "mathjax_ignore" e "tex2jax_ignore" evitam que o Jupyter leia "US$ ... $" como fórmula matemática
display(HTML(f'<div class="mathjax_ignore tex2jax_ignore">{tabela_html}</div>'))

Atributo,Valor
Nome,Carla Mendes (pessoa fictícia)
Cargo,Diretora de Engenharia
Relevância do cargo,9
Histórico profissional,"Não disponível/não verificável. O lead 'Carla Mendes' está explicitamente marcado como pessoa fictícia nos dados de cadastro (e-mail em domínio de exemplo 'exemplo.com.br'), portanto não há trajetória profissional pública confirmável. O cargo de Diretora de Engenharia é tratado como informação declarada no formulário do lead, e não como fato confirmado por fontes externas. A avaliação de relevância baseia-se no peso típico desse cargo (decisor/influenciador-chave técnico) dentro de uma organização de grande porte como o Nubank."
Empresa,Nubank (razão social: Nu Holdings Ltd.)
Setor,"Serviços Financeiros — Fintech / Banco digital (neobank). Atua em serviços financeiros e meios de pagamento, regulado e supervisionado pelo Banco Central do Brasil."
Porte (funcionários),10000
Faturamento,16300000000.000000
Presença de mercado,10
Nota do lead,94


## 15. O e-mail gerado

Este é o resultado final do pipeline: o e-mail personalizado e otimizado pelos dois agentes da crew de e-mail. O texto está em `.raw`.

In [15]:
import textwrap

if emails:
    for paragrafo in emails[0].raw.split("\n"):
        print(textwrap.fill(paragrafo, width=90))
else:
    print("Nenhum e-mail foi gerado: o lead não passou do filtro (nota > 70).")

Carla, você citou dois casos no formulário: automação de atendimento e análise de dados.
Em uma operação como a do Nubank — 140 milhões de clientes, quatro países, US$ 16,3
bilhões de receita, supervisão do Banco Central — isso não se resolve com um modelo
isolado. Resolve-se com orquestração.

**Quer ver isso desenhado dentro do Nubank?** Reserve 20 minutos aqui: [link de
agendamento].

A CrewAI orquestra múltiplos agentes de IA para rodar processos de ponta a ponta. Em vez
de um agente genérico, você monta times com papéis, ferramentas e regras próprias: um para
triagem de atendimento, outro para cruzar dados, outro para validar compliance. Cada
decisão fica rastreável.

Três pontos que pesam em times de engenharia como o seu:

- **Camada de orquestração, não substituto.** A CrewAI roda sobre os modelos que o Nubank
já usa, inclusive os da OpenAI.
- **Segurança, governança e auditoria** pensadas desde o início para ambientes regulados.
- **Escala sem inflar o time de engenharia.**

B

## 16. Indo além: fluxos com condições e desvios

Um Flow pode ser bem mais sofisticado. Nesta versão usamos mais três recursos:

- **`and_(a, b)`**: a etapa só roda quando **as duas** etapas `a` e `b` terminarem (aqui, `registrar_leads` espera o filtro **e** o armazenamento).
- **`or_(a, b)`**: roda quando **qualquer uma** terminar (disponível, mas não usado aqui).
- **`@router(etapa)`**: um "desvio". O método devolve um **rótulo** (texto) e só as etapas com `@listen("rótulo")` são executadas.

Nosso roteador `contar_leads` escolhe o caminho pela quantidade de leads aprovados:

| Quantidade | Rótulo | Caminho |
|---|---|---|
| mais de 10 | `"alto"` | enviar direto para o CRM (Salesforce) |
| de 6 a 10 | `"medio"` | repassar para o time de vendas |
| até 5 | `"baixo"` | escrever os e-mails com a crew de e-mail |

> **Dica:** a anotação `-> Literal['alto', 'medio', 'baixo']` no roteador informa ao CrewAI quais rótulos ele pode devolver. Assim o diagrama consegue desenhar os três caminhos.

> **Cuidado:** uma etapa que escuta um rótulo do roteador (`@listen('baixo')`) **não recebe os leads**. Ela é disparada pelo rótulo. Por isso `filtrar_leads` guarda os leads aprovados em `self.state["leads_aprovados"]`, e as etapas dos três caminhos leem de lá. O **estado** (`self.state`) é a "memória compartilhada" do fluxo.

In [16]:
from typing import Literal
from crewai.flow.flow import listen, start, and_, or_, router

class PipelineVendasCompleto(Flow):

    @start()
    def buscar_leads(self):
        # Exemplo fixo; num cenário real os leads viriam de um banco de dados
        leads = [
            {
                "dados_lead": {
                    "nome": "Carla Mendes (pessoa fictícia)",
                    "cargo": "Diretora de Engenharia",
                    "empresa": "Nubank",
                    "email": "carla.mendes@exemplo.com.br",
                    "caso_de_uso": "Usar agentes de IA para automatizar o atendimento ao cliente e a análise de dados.",
                },
            },
        ]
        return leads

    @listen(buscar_leads)
    def pontuar_leads(self, leads):
        resultados = crew_qualificacao_leads.kickoff_for_each(leads)
        self.state["resultados_qualificacao"] = resultados
        return resultados

    @listen(pontuar_leads)
    def armazenar_pontuacoes(self, resultados):
        # Aqui salvaríamos as notas no banco de dados
        return resultados

    @listen(pontuar_leads)
    def filtrar_leads(self, resultados):
        aprovados = [r for r in resultados if r['pontuacao_lead'].pontuacao > 70]
        self.state["leads_aprovados"] = aprovados   # guardado para os caminhos do roteador
        return aprovados

    @listen(and_(filtrar_leads, armazenar_pontuacoes))
    def registrar_leads(self, leads):
        print(f"Leads registrados: {len(leads)}")

    @router(filtrar_leads)
    def contar_leads(self, resultados) -> Literal['alto', 'medio', 'baixo']:
        if len(resultados) > 10:
            return 'alto'
        elif len(resultados) > 5:
            return 'medio'
        else:
            return 'baixo'

    # Atenção: quem escuta um rótulo do roteador recebe o RÓTULO (ex.: 'baixo'),
    # não os leads. Por isso os leads aprovados são lidos do self.state.

    @listen('alto')
    def armazenar_no_salesforce(self):
        return self.state["leads_aprovados"]

    @listen('medio')
    def enviar_para_time_de_vendas(self):
        return self.state["leads_aprovados"]

    @listen('baixo')
    def escrever_email(self):
        leads = self.state["leads_aprovados"]
        leads_pontuados = [lead.to_dict() for lead in leads]
        emails = crew_redacao_email.kickoff_for_each(leads_pontuados)
        return emails

    @listen(escrever_email)
    def enviar_email(self, emails):
        # Aqui enviaríamos os e-mails para os leads
        return emails

### Visualizando o fluxo completo

Compare com o desenho da seção 11: agora aparecem o roteador e os três caminhos possíveis.

In [17]:
fluxo_completo = PipelineVendasCompleto()
mostrar_fluxo(fluxo_completo, "fluxo_pipeline_vendas_completo")

### Executando o fluxo completo

Com um único lead, o roteador devolve `"baixo"` e o fluxo segue para a escrita do e-mail. Observe na saída a mensagem de `registrar_leads`, que só aparece depois que **as duas** etapas do `and_` terminam.

In [18]:
emails_completo = await fluxo_completo.kickoff_async()

if emails_completo:
    print("\n===== E-MAIL GERADO PELO FLUXO COMPLETO =====\n")
    for paragrafo in emails_completo[0].raw.split("\n"):
        print(textwrap.fill(paragrafo, width=90))

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: PipelineVendasCompleto                                                                                   │
│  ID: bf237ebc-5d39-46e7-9eb9-d81c0ccc6cea                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: PipelineVendasCompleto                                                                                   │
│  ID: bf237ebc-5d39-46e7-9eb9-d81c0ccc6cea                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: buscar_leads                                                                                           │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: buscar_leads                                                                                           │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: pontuar_leads                                                                                          │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: escrever_email                                                                                         │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: enviar_email                                                                                           │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: enviar_email                                                                                           │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: PipelineVendasCompleto                                                                                   │
│  ID: bf237ebc-5d39-46e7-9eb9-d81c0ccc6cea                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


===== E-MAIL GERADO PELO FLUXO COMPLETO =====

Você já provou que o Nubank consegue construir agentes — o Nu AI Engine é a prova. O
problema agora é outro: coordenar dezenas deles em produção, com contexto compartilhado,
handoffs e auditoria de cada decisão.

Criar um agente é fácil. Fazer 50 deles trabalharem juntos sem virar caos operacional —
isso ninguém resolveu direito ainda.

Foi por isso que você preencheu o formulário apontando dois casos de uso: automação de
atendimento e análise de dados com agentes de IA. Com mais de 100 milhões de clientes,
isso deixa de ser piloto e vira problema de escala.

É exatamente aí que a CrewAI entra.

A CrewAI é a camada de orquestração que faz seus agentes — os que vocês já construíram e
os que vão construir — trabalharem juntos. Conecta ao que vocês já têm (engine
proprietário, OpenAI, watsonx, Salesforce) e entrega enterprise-readiness: controle de
acesso, logs, rastreabilidade e deploy dentro do seu ambiente.

Para o Nubank, isso significa 

## 17. Resumo e exercícios

**O que vimos:**
- Agentes com papel, objetivo e história definidos em YAML.
- Ferramentas (busca no Google e leitura de sites) que dão aos agentes acesso à internet.
- Tarefas encadeadas com `context`, e saída estruturada com Pydantic.
- Duas crews trabalhando em sequência, coordenadas por um Flow.
- Flows avançados com `and_`, `or_` e `@router`.

**Sugestões de exercícios:**
1. Troque os dados de "nossa empresa e produto" nos arquivos YAML pela sua empresa ou por um produto inventado.
2. Acrescente mais leads na lista de `buscar_leads` e veja o `kickoff_for_each` processar todos.
3. Mude o corte do filtro (de 70 para outro valor) e observe o efeito.
4. Crie um **agente prospector** no início do fluxo que *descubra* leads na internet (por exemplo, "empresas de logística em São Paulo") em vez de usar uma lista fixa.
5. Troque o LLM no `.env` e compare a qualidade e o custo dos resultados.